# AIBackends - GLiNER2.5 advanced extraction and Decide classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-GLiNER2_5-advanced-extraction.ipynb)

Follow-up to the [GLiNER2.5 extraction notebook](AIBackends-GLiNER2_5-extraction.ipynb),
focused on the remaining `examples/gliner25/` scripts: spans longer than GLiNER2's
~12-word cap, multilingual NER with `gliner2.5-multi-v1`, clinical span attributes, and
multi-head operational decisions with the GLiNER2.5-Decide classifier.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
%pip install -q "aibackends[extraction]>=0.8.1"

# If a later import fails with a transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [2]:
import aibackends

# aibackends accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch

    DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print("aibackends", aibackends.__version__)
print("device:", DEVICE)

aibackends 0.8.1
device: cpu


## 1. Unlimited span width (`unlimited_spans.py`)

The boundary architecture scores span starts and ends, so a 25-word quote costs the same as a name.

In [3]:
from aibackends.tasks import extract_entities

TEXT = (
    'During the earnings call, CEO Amara Osei said "we expect revenue to grow by twenty '
    "percent next year driven by strong demand in our cloud division and continued "
    'expansion into Southeast Asian markets", before asking investors to send written '
    "questions to Meridian Cloud Holdings, Investor Relations, 4800 Lakeside Commons Drive, "
    "Suite 1200, Atlanta, Georgia 30339, United States."
)
LABELS = {
    "quote": "The complete quoted statement",
    "postal_address": "A complete postal address including street, city, and country",
    "person_name": "Full names of people",
}
result = extract_entities(TEXT, labels=LABELS, model="base", device=DEVICE, threshold=0.4)
for entity in result.entities:
    words = len(entity.text.split())
    flag = "  <- beyond GLiNER2's ~12-word cap" if words > 12 else ""
    print(f"[{entity.label}] {words} words{flag}\n  {entity.text}")
    assert TEXT[entity.start:entity.end] == entity.text

[person_name] 2 words
  Amara Osei
[quote] 25 words  <- beyond GLiNER2's ~12-word cap
  we expect revenue to grow by twenty percent next year driven by strong demand in our cloud division and continued expansion into Southeast Asian markets
[postal_address] 11 words
  4800 Lakeside Commons Drive, Suite 1200, Atlanta, Georgia 30339, United States


## 2. Multilingual NER (`multilingual_ner.py`)

English label names, non-English text, native batching.

In [4]:
from aibackends.tasks import extract_entities_batch

TEXTS = [
    "La empresa Iberdrola anunció una inversión de 3.000 millones de euros en Valencia "
    "junto a su presidente Ignacio Galán.",
    "Die Lufthansa eröffnet ein neues Drehkreuz in München, wie Vorstandschef Carsten "
    "Spohr am Montag erklärte.",
    "L'entreprise TotalEnergies a signé un accord avec le gouvernement du Sénégal à Dakar, "
    "selon Patrick Pouyanné.",
]
results = extract_entities_batch(
    TEXTS,
    labels=["person", "organization", "location", "monetary amount"],
    model="multi",
    device=DEVICE,
    threshold=0.4,
    batch_size=4,
)
for text, res in zip(TEXTS, results):
    print(text)
    for entity in res.entities:
        print(f"  [{entity.label}] {entity.text}")

La empresa Iberdrola anunció una inversión de 3.000 millones de euros en Valencia junto a su presidente Ignacio Galán.
  [organization] Iberdrola
  [monetary amount] 3.000 millones de euros
  [location] Valencia
  [person] Ignacio Galán
Die Lufthansa eröffnet ein neues Drehkreuz in München, wie Vorstandschef Carsten Spohr am Montag erklärte.
  [organization] Lufthansa
  [location] München
  [person] Carsten Spohr
L'entreprise TotalEnergies a signé un accord avec le gouvernement du Sénégal à Dakar, selon Patrick Pouyanné.
  [organization] TotalEnergies
  [organization] gouvernement du Sénégal
  [location] Dakar
  [person] Patrick Pouyanné


## 3. Clinical span attributes (`span_attributes.py`)

Negation and medication status are decoded in the same forward pass as the spans.

In [5]:
NOTE = (
    "Patient denies chest pain but reports severe headache and intermittent dizziness. "
    "Prescribed 400mg ibuprofen twice daily for the headache. Aspirin was discontinued "
    "last month due to a mild allergy."
)
result = extract_entities(
    NOTE,
    labels={
        "symptom": "Symptoms or complaints mentioned for the patient",
        "medication": "Names of drugs or pharmaceutical substances",
        "dosage": "Dose amounts such as 400mg or 2 tablets",
    },
    attributes={
        "negation": {"labels": ["present", "denied by patient"], "applies_to": ["symptom"]},
        "status": {
            "labels": ["currently prescribed", "discontinued"],
            "applies_to": ["medication"],
        },
    },
    model="base",
    device=DEVICE,
    threshold=0.4,
)
for entity in result.entities:
    qualifiers = ", ".join(f"{k}={v.label}" for k, v in entity.attributes.items())
    print(f"[{entity.label}] {entity.text}" + (f"  ({qualifiers})" if qualifiers else ""))

[symptom] chest pain  (negation=denied by patient)
[symptom] severe headache  (negation=present)
[symptom] intermittent dizziness  (negation=present)
[dosage] 400mg
[medication] ibuprofen  (status=currently prescribed)
[symptom] headache  (negation=present)
[medication] Aspirin  (status=discontinued)


## 4. Operational decisions with GLiNER2.5-Decide (`decide_classification.py`)

`model="decide"` loads the 340M classifier tuned for intent, routing, sentiment, priority,
and policy. Several heads run in one forward pass; heads can carry label descriptions,
a question (`prompt`), multi-label thresholds, or an ordinal scale.

In [6]:
from aibackends.tasks import classify_text, classify_texts

OPTIONS = {"model": "decide", "device": DEVICE}

review = (
    "The keyboard and the screen are the best I have used on a laptop, and the battery "
    "easily lasts a full workday."
)
result = classify_text(
    review,
    tasks={
        "sentiment": {"labels": ["positive", "negative", "mixed", "neutral"]},
        "aspects": {
            "labels": ["battery", "keyboard", "screen", "camera", "price", "support"],
            "multi_label": True,
            "cls_threshold": 0.4,
        },
    },
    **OPTIONS,
)
print("sentiment:", result.value("sentiment"), "| aspects:", result.values("aspects"))

pin = classify_text(
    "Please reset the card PIN. The new one never arrived and the old one is locked.",
    tasks={"intent": {"labels": {
        "card_pin_change": "The customer wants a new PIN or the current PIN replaced",
        "card_lost": "The physical card is missing",
        "balance_inquiry": "The customer wants the current balance",
    }}},
    **OPTIONS,
)
print("intent (with descriptions):", pin.value("intent"))

rating = classify_text(
    "Gave up after 40 pages. Flat characters and a plot you can see coming from the cover.",
    tasks={"rating": {"labels": [str(i) for i in range(11)], "ordinal": True}},
    **OPTIONS,
)
print("ordinal rating (0-10):", rating.value("rating"))

sentiment: positive | aspects: ['battery', 'keyboard', 'screen']


intent (with descriptions): card_pin_change


ordinal rating (0-10): 0


### Policy filter, then support routing

In [7]:
INBOX = [
    "Your mailbox is almost full. Click here in the next hour or we will delete every message.",
    "This is the third time I have explained the same missing refund. Get me a person.",
    "I was double charged this morning, please refund one of the payments.",
    "Tracking for my order hasn't moved since Monday.",
]
policies = classify_texts(
    INBOX,
    tasks={"policy": ["allow", "personal_data", "harassment", "scam", "violence", "spam"]},
    **OPTIONS,
)
allowed = [m for m, p in zip(INBOX, policies) if p.value("policy") == "allow"]
routes = classify_texts(
    allowed,
    tasks={
        "handoff": {
            "labels": ["yes", "no"],
            "prompt": "Should this conversation be handed off to a human agent?",
        },
        "intent": [
            "order_status", "refund_request", "cancel_subscription", "update_payment",
            "login_problem", "shipping_delay", "bug_report", "speak_to_human", "other",
        ],
        "urgency": ["low", "normal", "high", "critical"],
    },
    **OPTIONS,
)
decisions = dict(zip(allowed, routes))
for message, policy in zip(INBOX, policies):
    route = decisions.get(message)
    if route is None:
        action = f"block ({policy.value('policy')})"
    elif route.value("handoff") == "yes" or route.value("intent") == "speak_to_human":
        action = f"human_queue urgency={route.value('urgency')}"
    else:
        action = f"workflow:{route.value('intent')} urgency={route.value('urgency')}"
    print(f"{message[:60]:<62} -> {action}")

Your mailbox is almost full. Click here in the next hour or    -> block (spam)
This is the third time I have explained the same missing ref   -> human_queue urgency=high
I was double charged this morning, please refund one of the    -> workflow:refund_request urgency=normal
Tracking for my order hasn't moved since Monday.               -> workflow:order_status urgency=low
